# Config 1 time-resolved: electron and ion TOF vs delay (multi-file)

Loads one or more config-1 time-resolved aggregates
(`compute_aggregates.py` with `MODE = "time_resolved"`) and produces
side-by-side GMD-normalised eTOF / ion-TOF maps versus stage delay
`z` — one column per file.

The notebook follows the `delay_compare_runs.ipynb` convention:
specify a list of files in `FILES` (and an optional matching
`LABELS`), and every downstream cell loops over a `runs` dict so
adding or removing a file just adds or removes a column.

Three figures per loop:

1. Raw GMD-normalised spectrum vs delay (eTOF on the top row, ion
   TOF on the bottom row). If `DELAY_BASELINE` is set, the
   delay-averaged spectrum from that window is subtracted from every
   delay slice and a diverging colormap is used.
2. Same maps with a per-delay TOF-window baseline subtracted — the
   mean of the spectrum over a chosen TOF window (defined by two
   TOF endpoints) is subtracted from every TOF bin in the same
   delay slice. Removes a DC / dark offset. Stacks on top of the
   optional delay-baseline subtraction.
3. A region-of-interest zoom of the eTOF map per file, with both
   subtractions applied if set.

In [ ]:
import sys
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import config
from compute_aggregates import load_aggregates
%matplotlib inline

## Parameters

`FILES` is the list of config-1 time-resolved aggregate paths to
compare. `LABELS` is an optional matching list of short human
labels; when shorter than `FILES`, the missing entries fall back to
each file's stem.

`GMD_BIN` is a single integer applied to every file — every file
must have at least `GMD_BIN + 1` GMD bins. (If you need per-file
indices, replace `GMD_BIN` with `GMD_BINS = [...]` below.)

`ETOF_BASELINE_TOF` / `ION_TOF_BASELINE_TOF` give the TOF window
(in 100 ps units, same as `tof_edges`) used for plot 2's per-delay
DC baseline. `(None, None)` skips the subtraction.

`DELAY_BASELINE` (stage z units) selects a delay window whose
averaged spectrum is subtracted from every delay slice. `None`
skips. `ETOF_ROI` is the TOF window shown in plot 3; `(None, None)`
keeps the full range.

In [ ]:
# --- input --------------------------------------------------------
FILES = [
    config.COMBINED_DIR / "glycine_delay_scan_150C_272.0eV_aggregates_tr.h5",
]
# Optional human-readable labels (one per file). Missing entries
# fall back to the file stem.
LABELS = []

# --- GMD bin selection -------------------------------------------
GMD_BIN  = 1            # integer index, applied to every file

# --- TOF-window baseline ("two TOF bins") ------------------------
# Mean over this TOF window is subtracted from every TOF bin in the
# same delay slice. Units = 100 ps (matches tof_edges).
ETOF_BASELINE_TOF    = (None, None)   # e.g. (0.0, 500.0)
ION_TOF_BASELINE_TOF = (None, None)

# --- Delay-window baseline ---------------------------------------
# Mean spectrum over this delay window is subtracted from every
# delay slice. Set to None to skip. Units = stage z.
DELAY_BASELINE = (-5000, -1000)        # e.g. (-15000.0, -10000.0)

# --- eTOF zoom ---------------------------------------------------
ETOF_ROI = (None, None)                # e.g. (1000.0, 4000.0)

# --- cosmetics ---------------------------------------------------
PCT_LOW, PCT_HIGH = 1.0, 99.0          # robust colour-scale limits

print(f"files to compare: {len(FILES)}")
for p in FILES:
    print(f"  {p}")

## Load and normalise

Each file is loaded and validated as a config-1 time-resolved
aggregate. The `GMD_BIN` row is then GMD-normalised
(`agg.D[GMD_BIN] / agg.G[GMD_BIN][:, None]`) so each map is in
counts-per-shot per uJ and the delay axis isn't biased by fluence
drift within the GMD bin.

`runs` is a dict keyed by label; every downstream cell iterates
over `runs.items()`.

In [ ]:
def _label_for(i, path):
    if i < len(LABELS) and LABELS[i]:
        return LABELS[i]
    return Path(path).stem


def load_and_prepare(path):
    agg = load_aggregates(path)
    if agg.config != 1:
        raise ValueError(
            f"{Path(path).name}: expected config=1, got config={agg.config}"
        )
    if agg.mode != "time_resolved":
        raise ValueError(
            f"{Path(path).name}: expected mode='time_resolved', "
            f"got mode={agg.mode!r}"
        )
    if not (0 <= GMD_BIN < agg.n_gmd_bins):
        raise ValueError(
            f"{Path(path).name}: GMD_BIN={GMD_BIN} out of range "
            f"[0, {agg.n_gmd_bins})"
        )

    with np.errstate(invalid="ignore", divide="ignore"):
        D = agg.D[GMD_BIN] / agg.G[GMD_BIN][:, None]
        C = agg.C[GMD_BIN] / agg.G[GMD_BIN][:, None]

    gmd_label = (f"GMD [{agg.gmd_edges[GMD_BIN]:.3g}, "
                 f"{agg.gmd_edges[GMD_BIN + 1]:.3g}) uJ")
    return {
        "agg": agg,
        "D": D,
        "C": C,
        "tof_edges":     agg.tof_edges,
        "ion_tof_edges": agg.ion_tof_edges,
        "z_edges":       agg.z_edges,
        "gmd_label":     gmd_label,
        "n_tof":   agg.n_tof,
        "n_tof_i": agg.n_tof_i,
        "n_z":     agg.n_z_bins,
        "n_per_bin": agg.n_per_bin[GMD_BIN],
    }


runs = {_label_for(i, p): load_and_prepare(p) for i, p in enumerate(FILES)}

for label, run in runs.items():
    print(f"{label}:")
    print(f"  GMD       : {run['gmd_label']}")
    print(f"  n_z       : {run['n_z']}")
    print(f"  n_tof (e) : {run['n_tof']}")
    print(f"  n_tof (i) : {run['n_tof_i']}")
    print(f"  shots/bin : min={run['n_per_bin'].min()}, "
          f"max={run['n_per_bin'].max()}, "
          f"total={run['n_per_bin'].sum()}")

## Subtraction helpers

`_subtract_tof_baseline`: per-delay mean over a TOF window,
subtracted from every TOF bin in the same delay slice.

`_subtract_delay_baseline`: mean spectrum over a delay window,
subtracted from every delay slice. Pass `None` or `(None, None)` to
skip either.

`_plot_panel` paints one map (pcolormesh) into a supplied axis, with
either a robust viridis scale (raw) or a symmetric diverging
RdBu_r / `TwoSlopeNorm` scale (after any subtraction).

In [ ]:
def _window_indices(edges, lo, hi):
    """Index range [i_lo, i_hi) of bins whose centres fall in [lo, hi)."""
    cent = 0.5 * (edges[:-1] + edges[1:])
    if lo is None:
        lo = cent[0]
    if hi is None:
        hi = cent[-1] + np.finfo(float).eps
    sel = (cent >= lo) & (cent < hi)
    idx = np.where(sel)[0]
    if idx.size == 0:
        raise ValueError(
            f"empty window [{lo}, {hi}) for edges spanning "
            f"[{edges[0]}, {edges[-1]})"
        )
    return int(idx[0]), int(idx[-1] + 1)


def _subtract_tof_baseline(spec, tof_edges, tof_window):
    if tof_window is None or tof_window == (None, None):
        return spec
    lo, hi = tof_window
    if lo is None and hi is None:
        return spec
    i0, i1 = _window_indices(tof_edges, lo, hi)
    base = np.nanmean(spec[:, i0:i1], axis=1, keepdims=True)
    return spec - base


def _subtract_delay_baseline(spec, z_edges, delay_window):
    if delay_window is None:
        return spec
    lo, hi = delay_window
    if lo is None and hi is None:
        return spec
    j0, j1 = _window_indices(z_edges, lo, hi)
    base = np.nanmean(spec[j0:j1, :], axis=0, keepdims=True)
    return spec - base


def _robust_limits(arr, pct_lo, pct_hi, symmetric=False):
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return -1.0, 1.0
    if symmetric:
        lim = float(np.nanpercentile(np.abs(finite), pct_hi))
        if lim <= 0:
            lim = float(np.nanmax(np.abs(finite))) or 1.0
        return -lim, lim
    return (float(np.nanpercentile(finite, pct_lo)),
            float(np.nanpercentile(finite, pct_hi)))


def _plot_panel(ax, spec, x_edges, y_edges, title, xlabel, cbar_label,
                symmetric):
    if symmetric:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH, symmetric=True)
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="RdBu_r",
                            norm=norm, shading="auto")
    else:
        vmin, vmax = _robust_limits(spec, PCT_LOW, PCT_HIGH)
        pcm = ax.pcolormesh(x_edges, y_edges, spec, cmap="viridis",
                            vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    cbar = ax.figure.colorbar(pcm, ax=ax)
    cbar.set_label(cbar_label)
    return pcm


def _tag(window, name):
    if window is None or window == (None, None):
        return f"no {name}"
    return f"{name} [{window[0]}, {window[1]})"


def _delay_tag():
    return (f", delay [{DELAY_BASELINE[0]}, {DELAY_BASELINE[1]})"
            if DELAY_BASELINE is not None else "")

## Plot 1 — raw GMD-normalised spectrum vs delay

Top row: eTOF map per file. Bottom row: ion TOF map per file. If
`DELAY_BASELINE` is set, the spectrum averaged over that delay
window is subtracted from every delay slice (diverging colormap).
Otherwise the raw GMD-normalised spectrum is shown (viridis).

In [ ]:
sym1 = DELAY_BASELINE is not None
suffix = "  - delay-baseline" if sym1 else ""
n_files = len(runs)

fig, axes = plt.subplots(2, n_files,
                         figsize=(6 * n_files, 8),
                         sharey="row", constrained_layout=True,
                         squeeze=False)
for k, (label, run) in enumerate(runs.items()):
    D_p1 = _subtract_delay_baseline(run["D"], run["z_edges"], DELAY_BASELINE)
    C_p1 = _subtract_delay_baseline(run["C"], run["z_edges"], DELAY_BASELINE)

    _plot_panel(axes[0, k], D_p1, run["tof_edges"], run["z_edges"],
                f"{label}  -  eTOF{suffix}",
                "eTOF (100 ps)", "D / G (counts/shot/uJ)", sym1)
    _plot_panel(axes[1, k], C_p1, run["ion_tof_edges"], run["z_edges"],
                f"{label}  -  ion TOF{suffix}",
                "ion TOF (100 ps)", "C / G (counts/shot/uJ)", sym1)

axes[0, 0].set_ylabel("stage z (arb.)")
axes[1, 0].set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"GMD-normalised TOF vs delay  -  {gmd_labels}")
plt.show()

## Plot 2 — TOF-window baseline subtracted

For each delay slice the mean of the spectrum over the chosen TOF
window is subtracted from every TOF bin. Use this to remove a
delay-dependent DC offset (dark / scatter floor). If
`DELAY_BASELINE` is also set, the delay-averaged spectrum is
subtracted on top.

In [ ]:
n_files = len(runs)

fig, axes = plt.subplots(2, n_files,
                         figsize=(6 * n_files, 8),
                         sharey="row", constrained_layout=True,
                         squeeze=False)
for k, (label, run) in enumerate(runs.items()):
    D_p2 = _subtract_tof_baseline(run["D"], run["tof_edges"],
                                  ETOF_BASELINE_TOF)
    C_p2 = _subtract_tof_baseline(run["C"], run["ion_tof_edges"],
                                  ION_TOF_BASELINE_TOF)
    D_p2 = _subtract_delay_baseline(D_p2, run["z_edges"], DELAY_BASELINE)
    C_p2 = _subtract_delay_baseline(C_p2, run["z_edges"], DELAY_BASELINE)

    _plot_panel(
        axes[0, k], D_p2, run["tof_edges"], run["z_edges"],
        f"{label}  -  eTOF\n{_tag(ETOF_BASELINE_TOF, 'TOF baseline')}{_delay_tag()}",
        "eTOF (100 ps)", "(D - baseline) / G", symmetric=True,
    )
    _plot_panel(
        axes[1, k], C_p2, run["ion_tof_edges"], run["z_edges"],
        f"{label}  -  ion TOF\n{_tag(ION_TOF_BASELINE_TOF, 'TOF baseline')}{_delay_tag()}",
        "ion TOF (100 ps)", "(C - baseline) / G", symmetric=True,
    )

axes[0, 0].set_ylabel("stage z (arb.)")
axes[1, 0].set_ylabel("stage z (arb.)")
gmd_labels = " | ".join(sorted({r["gmd_label"] for r in runs.values()}))
fig.suptitle(f"TOF-baseline subtracted  -  {gmd_labels}")
plt.show()

## Plot 3 — eTOF region-of-interest vs delay

Zooms the eTOF map into `ETOF_ROI`. All subtractions configured
above (TOF baseline + optional delay baseline) are applied to each
file independently.

In [ ]:
n_files = len(runs)
sym_roi = (ETOF_BASELINE_TOF != (None, None)) or (DELAY_BASELINE is not None)

fig, axes = plt.subplots(1, n_files,
                         figsize=(6 * n_files, 5),
                         sharey=True, constrained_layout=True,
                         squeeze=False)

for k, (label, run) in enumerate(runs.items()):
    D_roi = _subtract_tof_baseline(run["D"], run["tof_edges"],
                                   ETOF_BASELINE_TOF)
    D_roi = _subtract_delay_baseline(D_roi, run["z_edges"], DELAY_BASELINE)

    roi_lo, roi_hi = ETOF_ROI
    if roi_lo is None and roi_hi is None:
        i0, i1 = 0, run["n_tof"]
    else:
        i0, i1 = _window_indices(run["tof_edges"], roi_lo, roi_hi)
    print(f"{label}: eTOF ROI bins [{i0}, {i1}) -> "
          f"[{run['tof_edges'][i0]:.1f}, {run['tof_edges'][i1]:.1f}) 100 ps")

    D_roi_slice = D_roi[:, i0:i1]
    roi_edges   = run["tof_edges"][i0:i1 + 1]

    _plot_panel(
        axes[0, k], D_roi_slice, roi_edges, run["z_edges"],
        f"{label}  -  {run['gmd_label']}",
        "eTOF (100 ps)",
        "(D - baseline) / G" if sym_roi else "D / G (counts/shot/uJ)",
        symmetric=sym_roi,
    )

axes[0, 0].set_ylabel("stage z (arb.)")
fig.suptitle("eTOF ROI vs delay")
plt.show()